# Lab type: debug
# Course: ML401 — MLOps & Model Deployment
# Lesson: Containerising ML Models
# Task: The Dockerfile and docker-compose.yml below each contain production issues. Identify all issues, explain the failure each causes, and write corrected versions.

## The broken Dockerfile

Your team has asked you to review this Dockerfile before it is used to build the production model serving image.
There are **4 issues** in this file. Identify each one before looking at the analysis.

```dockerfile
FROM python:latest

WORKDIR /app

COPY . .

RUN pip install -r requirements.txt

EXPOSE 8080

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8080"]
```

## Your Dockerfile analysis

**Issue 1:**
- Location in file:
- What is wrong:
- Failure mode in production:

**Issue 2:**
- Location in file:
- What is wrong:
- Failure mode in production:

**Issue 3:**
- Location in file:
- What is wrong:
- Failure mode in production:

**Issue 4:**
- Location in file:
- What is wrong:
- Failure mode in production:

## Dockerfile analysis: answers

**Issue 1: `python:latest` — unpinned base image.**
`latest` resolves at build time. An image built today and an image built in six months may use different Python versions, different base OS packages, and different system library versions. The artefact is not reproducible. Fix: `FROM python:3.11.9-slim`.

**Issue 2: `COPY . .` before `pip install` — layer cache defeated.**
Docker caches each layer. When source code changes (every deploy), the `COPY . .` layer is invalidated, forcing `pip install` to run from scratch on every build. Separating `COPY requirements.txt` + `pip install` from `COPY . .` restores caching: dependencies are only reinstalled when `requirements.txt` changes.

**Issue 3: No non-root user.**
The container process runs as root. If the application is compromised, the attacker has root-level access to the container filesystem. Most container security policies (Kubernetes Pod Security Admission, OPA Gatekeeper) will block root containers in production namespaces.

**Issue 4: No HEALTHCHECK.**
Without a health check, Docker and Kubernetes consider the container healthy as soon as the process starts. A model server that starts but fails to load its artefact will receive traffic. A health check endpoint at `/health` that verifies the model is loaded is the minimum requirement.

## Write the corrected Dockerfile

Write a corrected Dockerfile in the cell below that fixes all four issues.
The corrected file should:
- Use a pinned Python base image
- Install dependencies before copying source code
- Run as a non-root user named `appuser`
- Include a HEALTHCHECK with appropriate startup period for a model that takes ~45 seconds to load

```dockerfile
# Write your corrected Dockerfile here:


```

## Corrected Dockerfile: answer

```dockerfile
FROM python:3.11.9-slim

# Create non-root user
RUN useradd --create-home appuser
WORKDIR /home/appuser

# Install dependencies first (cached layer)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy source code after dependencies
COPY --chown=appuser:appuser . .

# Switch to non-root user
USER appuser

EXPOSE 8080

# Health check — start-period accounts for 45s model load time
HEALTHCHECK --interval=30s --timeout=10s --start-period=60s --retries=3 \
  CMD curl -f http://localhost:8080/health || exit 1

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8080"]
```

## The broken docker-compose.yml

The following `docker-compose.yml` is used for local integration testing.
There are **3 issues** that will cause problems when this configuration is used as a reference for production deployment.

```yaml
version: '3.8'

services:
  model-api:
    build: .
    ports:
      - "8080:8080"
    environment:
      - MODEL_PATH=./models/churn_model.joblib
      - SECRET_KEY=dev-secret-key-do-not-use-in-production
    volumes:
      - ./models:/app/models
      - .:/app  # mount source code for hot reload
    restart: always
```

## Your docker-compose analysis

**Issue 1:**
- What is wrong:
- Why this is a problem if this configuration is referenced for production:

**Issue 2:**
- What is wrong:
- Why this is a problem if this configuration is referenced for production:

**Issue 3:**
- What is wrong:
- Why this is a problem if this configuration is referenced for production:

## docker-compose analysis: answers

**Issue 1: Hardcoded secret key in environment variable.**
`SECRET_KEY=dev-secret-key-do-not-use-in-production` is stored in plaintext in a file that will be committed to the repository. In production, secrets must come from a secrets manager (Kubernetes Secrets, AWS Secrets Manager, Vault) — not from environment variables in config files checked into source control.

**Issue 2: Source code volume mount (`. :/app`).**
Mounting the source code directory for hot-reload is a development convenience, not a production pattern. In production, the container image must be the single source of truth for the application code. A live volume mount bypasses the build process and creates untested code paths.

**Issue 3: `restart: always` without a readiness gate.**
In production orchestration (Kubernetes), restart behaviour is managed by the liveness probe — not by a restart policy that restarts on any exit. `restart: always` will restart a container that exits due to a startup failure (e.g., model artefact not found) in an infinite loop, consuming resources without resolving the underlying error. The correct approach is to ensure the startup error is surfaced and the loop is broken by the orchestrator's backoff policy.

## Reflection

1. A colleague argues that because `docker-compose.yml` is only used locally, the issues above don't matter. What is the risk of this reasoning in practice?

2. If your model artefact is 2GB and must be included in the Docker image, how does this affect your build strategy? What alternatives exist to including the artefact directly in the image?